# Ejercicio 03d  |  Tema 3M - PRyES
## Distribución discreta bivariante

stats

In [ ]:
# import required packages
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd

### Se tiene un vector aleatorio [X, Y] que sigue la distribución bivariante indicada mediante su función de masa de probabilidad conjunta pmfXY:

In [ ]:
# v.a. bidimensional de partida

# valores posibles de la v.a. X e Y
xk = list(range(1,25))
yk = list(range(0,11))

# probabilidad de cada pareja (xk, yk), xk: columnas, yk: filas
# leemos los datos de la pmf del fichero usando pandas
# y lo convierte a una matriz
df_orig = pd.read_csv('jpmf_E03d.txt', header=None)
pmfXY = np.matrix(df_orig.to_numpy())

print('xk:', xk)
print('yk:', yk)
print('Función de masa de probabilidad conjunta:\n', pmfXY)

### a) Representa la función de masa de probabilidad conjunta pmfXY en una gráfica 3D.

In [ ]:
# Representación de la pmf conjunta

# muestra SUPERFICIE 3D de la pmf
X, Y = np.meshgrid(xk,yk)

# evalua la pdf en cada punto (x_i,y_j)
Z = np.zeros(X.shape)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
         Z[i,j] = pmfXY[i,j]

fig = plt.figure(2, figsize=(8, 8))
ax = plt.axes(projection='3d')
ax.set_xticks(xk)
ax.set_yticks(yk)
ax.invert_xaxis()
ax.stem(X.flatten(), Y.flatten(), Z.flatten(), basefmt='none')
plt.xlabel('x')
plt.ylabel('y')
plt.title('pmf conjunta')

# cambia orientación para que se vea bien
ax.azim = -45
ax.dist = 10
ax.elev = 70
plt.show()

### b) Calcula la función de distribución acumulada conjunta de (X,Y) y represéntala gráficamente.

In [ ]:
# Cálculo

# función que crea la cdfXY (es una matriz con los valores acumulados)
def crea_matriz_cdfXY(pmfXY): # crea una matriz a partir de la info de la V.A.
    # sumas acumuladas horizontales y luego las acumula verticalmente
    jcdf = np.cumsum(np.cumsum(pmfXY,axis=1),axis=0)
    return jcdf

# crea la función de probabilidad conjunta como una matriz
cdfXY = crea_matriz_cdfXY(pmfXY)

In [ ]:
# Gráficamente

# obtiene los puntos de cambio primero
X, Y = np.meshgrid(xk,yk)
# evalua la cdf en cada punto (x_i,y_j)
Z = np.zeros(X.shape)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
         Z[i,j] = cdfXY[i,j]

# ahora los puntos para la superficie
xk1 = np.append(xk, xk[-1] + (xk[-1]-xk[-2]) )
yk1 = np.append(yk, yk[-1] + (yk[-1]-yk[-2]) )

xxs = np.linspace(xk[0]-0.5*(xk[1]-xk[0]), xk1[-1]-0.01, 500)
yys = np.linspace(yk[0]-0.5*(yk[1]-yk[0]), yk1[-1]-0.01, 500)
XX, YY = np.meshgrid(xxs,yys)
ZZ = np.zeros(XX.shape)

for i in range(len(xk1)):
    for j in range(len(yk1)):
        ind_xxs = (XX < xk1[i]) & (XX >= xk1[i-1])
        ind_yys = (YY < yk1[j]) & (YY >= yk1[j-1])
        ind_zzs = ind_xxs & ind_yys
        ZZ[ind_zzs] = cdfXY[j-1,i-1]
        
# representa
fig = plt.figure(1, figsize=(18, 8))
ax = fig.add_subplot(121,projection='3d')   
ax.plot_surface(XX, YY, ZZ, alpha=0.95, rstride=2, cstride=2, cmap='viridis', edgecolor='none')
ax.stem(X.flatten(), Y.flatten(), Z.flatten(), basefmt='none')

ax.set_xticks(xk)
ax.set_yticks(yk)
ax.invert_xaxis()
plt.xlabel("X")
plt.ylabel("Y")
plt.title('cdf conjunta')
ax.azim = -50
ax.dist = 10
ax.elev = 40

# representa NIVELES en el otro eje
ax = fig.add_subplot(122)   
ax.set_xticks(xk)
ax.set_yticks(yk)
ax.invert_xaxis()
plt.contourf(XX, YY, ZZ, cmap='viridis', alpha = 1) # alpha para controlar la transparencia y poder pintar varias

plt.xlabel("X")
plt.ylabel("Y")
plt.title('cdf conjunta (niveles)')
plt.grid()
plt.colorbar()
plt.show()

### c) Calcula la probabilidad de que X sea igual a 14.

In [ ]:
# Respuesta

print('Para calcular P(X=14) se necesita la pmf de la marginal X.')

pmfX = np.sum(pmfXY, 0).getA1()
VA_X = stats.rv_discrete(values=(xk, pmfX))
print('Verifica que 1 = suma de Marginal pmfX = ', pmfX.sum())
print('Marginal pmfX:', pmfX)

# usamos la v.a. creada
prob = VA_X.pmf(14)
print(f'Usando la v.a. X se obtiene: P(X=14) = X.pmf(14) = {prob}')

### d) Calcula la probabilidad de que Y sea menor o igual que 24.

In [ ]:
# Respuesta

print('Para calcular P(Y<=24) se necesita la cdf de la marginal Y.')

# Marginal Y a partir de la pmf de (X,Y) discreta bivariante
pmfY = np.sum(pmfXY, 1).getA1()
VA_Y = stats.rv_discrete(values=(yk, pmfY))
print('Verifica que 1 = suma de Marginal pmfY = ', pmfY.sum())
print('Marginal pmfY:', pmfY)

# usamos la v.a. creada
prob = VA_Y.cdf(24)
print(f'Usando la v.a. Y se obtiene: P(Y<=24) = Y.cdf(24) = {prob}, un valor lógico ya que los valores de Y están entre 0 y 10.')

### e) Representa las distribuciones marginales de X e Y, calculando los valores esperados de dichas v.a. ¿Es coherente con lo que observas en las gráficas marginales?

In [ ]:
# Respuesta

# Representa marginales X e Y
fig = plt.figure(1, figsize=(10, 4))

plt.subplot(1,2,1)
plt.stem(xk, pmfX, basefmt='none')
plt.ylim(0, 0.25)
plt.xlabel('x')
plt.ylabel('P(X=x)')
plt.title('Marginal X');plt.grid()

plt.subplot(1,2,2)
plt.stem(yk, pmfY, basefmt='none')
plt.ylim(0, 0.25)
plt.xlabel('y')
plt.ylabel('P(Y=y)')
plt.title('Marginal Y');plt.grid()

plt.show()

# calcula los valores esperados

media_X = VA_X.mean()
media_Y = VA_Y.mean()
print(f'El valor esperado de X es {media_X}, y el de Y vale {media_Y}. Son valores lógicos, viendo la forma de las distribuciones de masa de '
      'probabilidad de las marginales.')

### f) Calcula y representa la función de masa de probabilidad de X condicionada a que Y = 4. Utiliza dicha función para calcular la P(X=15 | Y=4). 

In [ ]:
# Respuesta

# Condicionada X | Y = y para (X,Y) discreta bivariante
y = 4
fila_valor_y = yk.index(y)
print(f'Posición del valor y={y} en yk: {fila_valor_y}')

pmfXcY = (pmfXY[fila_valor_y,:] / pmfY[fila_valor_y]).getA1()
print(f'Verifica que 1 = suma de pmfX|Y={y} = ', pmfXcY.sum())
print(f'pmfX|Y={y} = {pmfXY[fila_valor_y,:]}/{pmfY[fila_valor_y]} = {pmfXcY}')
VA_XcondY = stats.rv_discrete(values=(xk, pmfXcY))

# representa la condicionada
fig = plt.figure(1, figsize=(10, 4))
plt.stem(xk, pmfXcY, basefmt='none')
plt.ylim(0, 0.2)
plt.xlabel('x')
plt.ylabel(f'P(X=x|Y={y})')
plt.title(f'Condicionada X|Y={y}'); plt.grid()

# Cálculo pedido
print('Para calcular P(X=15|Y=4) se necesita la pmf de la v.a X|Y=4.')
prob = VA_XcondY.pmf(15)
print(f'Usando la v.a. X|Y=4 se obtiene: P(X=15|Y=4) = X|Y=4.cdf(15) = {prob}')

### g) Observando la función de masa de probabilidad de la v.a. unidimensional X|Y=4, representada en el apartado anterior, imagina su valor esperado y confirma tu estimación cálculando el mismo a partir de la función de distribución de dicha v.a. Haz lo mismo con el percentil 25.

In [ ]:
# Respuesta

print('La distribución de X|Y=4 parece simétrica, por tanto, E[X|Y=4] debería estar en el centro de los valores que puede tomar X, '
      'es decir, 12.5. Teniendo en cuenta que hay 24 valores y la simetría de la función, el percentil 25 podría ser 5 o algo así.')

# valor esperado
media = VA_XcondY.mean()
print(f'Usando la v.a. X|Y=4 se obtiene: E[X|Y=4] = {media}')

# percentil 25 usando la inversa de la cdf
q = 0.25
p25 = VA_XcondY.ppf(q)
print(f'Usando la inversa de la cdf de la v.a. X|Y=4 se obtiene: percentil 25 = {p25}')

### h) Calcula el valor esperado de Y condicionado a cada uno de los diferentes valores de X. Relaciona estos valores con la esperanza matemática de Y.

In [ ]:
# Respuesta

print('El valor esperado de Y|X=x se obtiene calculando la esperanza de la correspondiente distribución condicionada.')

medias = []
for x in xk:
    colum_valor_x = xk.index(x)
    pmfYcX_x = (pmfXY[:, colum_valor_x] / pmfX[colum_valor_x]).getA1()
    VA_YcondX_x = stats.rv_discrete(values=(yk, pmfYcX_x))
    media_fija_x = VA_YcondX_x.mean()
    medias.append(media_fija_x)

print(f'El valor esperado de Y según el valor de x que condiciona es: {medias}')

media_medias = np.mean(medias)
media_Y = VA_Y.mean()

print('La esperanza matemática de Y, usando de la esperanza total, vale E[Y] = E[E[Y|X]]. Se puede comprobar fácilmente que la igualdad es cierta: ')
print(f'   E[E[Y|X]] = media de las medias = {media_medias}')
print(f'   E[Y] = media de Y = {media_Y}')

# gráficamente
fig = plt.figure(1, figsize=(10, 4))
plt.bar(xk, medias, label = 'E[Y|X=x]')
plt.hlines(media_Y,xk[0],25,color = 'red', label = 'E[Y] = E[E[Y|X]]')
plt.xlabel('x')
plt.legend()
plt.show()